# NMF on ModulAir 00678

In [22]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [23]:
#importing data from Modulair MOD-00678
data = pd.read_csv('MOD-00678.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:38Z,577612587,2025-12-31T18:59:38Z,MOD-00678,49.5,0.4,7.221,0.804,0.196,0.017,0.037,...,27.638,34.986,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.24
2025-12-31T23:58:38Z,577610602,2025-12-31T18:58:38Z,MOD-00678,49.1,0.3,6.655,0.625,0.113,0.030,0.016,...,25.989,37.115,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.28
2025-12-31T23:57:38Z,577610603,2025-12-31T18:57:38Z,MOD-00678,49.3,0.3,6.896,0.719,0.185,0.018,0.023,...,25.507,36.403,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.50
2025-12-31T23:56:38Z,577610601,2025-12-31T18:56:38Z,MOD-00678,49.6,0.3,7.017,0.791,0.154,0.019,0.026,...,25.968,36.735,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.13
2025-12-31T23:55:38Z,577610600,2025-12-31T18:55:38Z,MOD-00678,49.6,0.3,6.525,0.657,0.140,0.024,0.019,...,26.916,35.683,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,0.90


In [24]:
start = data.index.min()
end = data.index.max()

print(start, end)

2025-04-15T12:20:37Z 2025-12-31T23:59:38Z


In [25]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:38Z,2025-12-31T18:59:38Z,762.896,2.403,27.638,34.986,7.221,0.804,0.196,0.017,0.037,0.012
2025-12-31T23:58:38Z,2025-12-31T18:58:38Z,752.810,2.402,25.989,37.115,6.655,0.625,0.113,0.030,0.016,0.020
2025-12-31T23:57:38Z,2025-12-31T18:57:38Z,766.214,2.468,25.507,36.403,6.896,0.719,0.185,0.018,0.023,0.017
2025-12-31T23:56:38Z,2025-12-31T18:56:38Z,760.254,2.467,25.968,36.735,7.017,0.791,0.154,0.019,0.026,0.011
2025-12-31T23:55:38Z,2025-12-31T18:55:38Z,756.560,2.402,26.916,35.683,6.525,0.657,0.140,0.024,0.019,0.008


In [26]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:38Z,762.896,2.403,27.638,34.986,7.221,0.804,0.196,0.017,0.037,0.012
1,2025-12-31T18:58:38Z,752.810,2.402,25.989,37.115,6.655,0.625,0.113,0.030,0.016,0.020
2,2025-12-31T18:57:38Z,766.214,2.468,25.507,36.403,6.896,0.719,0.185,0.018,0.023,0.017
3,2025-12-31T18:56:38Z,760.254,2.467,25.968,36.735,7.017,0.791,0.154,0.019,0.026,0.011
4,2025-12-31T18:55:38Z,756.560,2.402,26.916,35.683,6.525,0.657,0.140,0.024,0.019,0.008


In [27]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:38,762.896,2.403,27.638,34.986,7.221,0.804,0.196,0.017,0.037,0.012
1,2025-12-31 18:58:38,752.810,2.402,25.989,37.115,6.655,0.625,0.113,0.030,0.016,0.020
2,2025-12-31 18:57:38,766.214,2.468,25.507,36.403,6.896,0.719,0.185,0.018,0.023,0.017
3,2025-12-31 18:56:38,760.254,2.467,25.968,36.735,7.017,0.791,0.154,0.019,0.026,0.011
4,2025-12-31 18:55:38,756.560,2.402,26.916,35.683,6.525,0.657,0.140,0.024,0.019,0.008


In [28]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
2,2025-04-15 14:00:00,933.86,7.56,68.75,3.40,3.78,0.45,0.17,0.05,0.07,0.06
3,2025-04-15 15:00:00,829.55,7.11,68.36,3.70,2.20,0.33,0.14,0.05,0.08,0.06
4,2025-04-15 16:00:00,796.16,15.65,65.49,2.66,1.84,0.28,0.13,0.05,0.07,0.05
5,2025-04-15 17:00:00,807.97,9.88,63.24,2.64,2.24,0.30,0.12,0.04,0.05,0.04
6,2025-04-15 18:00:00,856.53,11.27,63.59,2.68,2.00,0.25,0.09,0.02,0.03,0.02
...,...,...,...,...,...,...,...,...,...,...,...
6209,2025-12-31 14:00:00,736.98,23.61,42.09,2.03,9.58,1.29,0.44,0.13,0.20,0.16
6210,2025-12-31 15:00:00,741.31,24.45,41.62,1.88,8.94,1.16,0.38,0.10,0.14,0.10
6211,2025-12-31 16:00:00,760.21,25.09,40.17,2.20,8.79,0.86,0.23,0.04,0.04,0.02
6212,2025-12-31 17:00:00,775.26,26.34,37.12,2.08,8.00,0.78,0.17,0.02,0.03,0.01


In [29]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 14:00:00,933.86,7.56,68.75,3.40,3.78,0.45,0.17,0.05,0.07,0.06
2025-04-15 15:00:00,829.55,7.11,68.36,3.70,2.20,0.33,0.14,0.05,0.08,0.06
2025-04-15 16:00:00,796.16,15.65,65.49,2.66,1.84,0.28,0.13,0.05,0.07,0.05
2025-04-15 17:00:00,807.97,9.88,63.24,2.64,2.24,0.30,0.12,0.04,0.05,0.04
2025-04-15 18:00:00,856.53,11.27,63.59,2.68,2.00,0.25,0.09,0.02,0.03,0.02


In [30]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [31]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [32]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 14:00:00,0.280959,0.192808,0.758663,0.080952,0.048617,0.013971,0.009901,0.010661,0.018135,0.024490
2025-04-15 15:00:00,0.249577,0.181331,0.754359,0.088095,0.028296,0.010245,0.008154,0.010661,0.020725,0.024490
2025-04-15 16:00:00,0.239531,0.399133,0.722688,0.063333,0.023666,0.008693,0.007571,0.010661,0.018135,0.020408
2025-04-15 17:00:00,0.243084,0.251977,0.697859,0.062857,0.028810,0.009314,0.006989,0.008529,0.012953,0.016327
2025-04-15 18:00:00,0.257694,0.287427,0.701721,0.063810,0.025723,0.007762,0.005242,0.004264,0.007772,0.008163


In [33]:
len(data)

6146

In [ ]:
data.to_csv(r'MOD-00678_timeseries_hourly_scaled.csv')

## Implementing NMF

In [34]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [35]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 14:00:00,0.261393,0.197049,0.764786,0.064375,0.054295,0.006093,0.005150,0.009254,0.023467,0.033634
2025-04-15 15:00:00,0.254425,0.180492,0.753110,0.062986,0.051254,0.004983,0.004548,0.008728,0.022719,0.032720
2025-04-15 16:00:00,0.283303,0.389597,0.707746,0.064977,0.053100,0.002876,0.002624,0.006742,0.019483,0.028755
2025-04-15 17:00:00,0.251082,0.249968,0.694026,0.060117,0.049114,0.001263,0.001153,0.005175,0.017467,0.026480
2025-04-15 18:00:00,0.259778,0.286505,0.698946,0.061409,0.050244,0.000000,0.000000,0.004116,0.016265,0.025265
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.261355,0.594262,0.454096,0.053215,0.112835,0.054059,0.034302,0.031133,0.042002,0.049212
2025-12-31 15:00:00,0.262548,0.615543,0.447959,0.052718,0.108265,0.041531,0.023859,0.021398,0.030427,0.036938
2025-12-31 16:00:00,0.261120,0.633438,0.433878,0.051257,0.110953,0.024038,0.007231,0.005167,0.010724,0.015947


In [36]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.079037,0.000482,0.017322,0.003991
1,0.077917,0.000000,0.014681,0.003644
2,0.073224,0.000000,0.052671,0.002103
3,0.071805,0.000000,0.028350,0.000924
4,0.072314,0.000000,0.034710,0.000000
...,...,...,...,...
6141,0.043747,0.017674,0.093780,0.022492
6142,0.043325,0.016510,0.097872,0.014453
6143,0.041725,0.017289,0.101622,0.000908
6144,0.038807,0.015356,0.108301,0.000000


In [37]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [38]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,3.044598,1.243900,9.665484,0.776646,0.633890,0.000000,0.000000,0.055413,0.224679,0.349377
Feature 2,1.041090,0.345105,1.768875,0.198160,4.141820,1.318625,0.352748,0.083739,0.000000,0.000000
Feature 3,1.141223,5.662767,0.000000,0.151175,0.126908,0.000000,0.000000,0.003152,0.000516,0.000000
Feature 4,0.121766,0.119568,0.000000,0.069349,0.000000,1.367277,1.247841,1.197437,1.428245,1.508435


In [39]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-04-15 14:00:00,0.079037,0.000482,0.017322,0.003991
2025-04-15 15:00:00,0.077917,0.000000,0.014681,0.003644
2025-04-15 16:00:00,0.073224,0.000000,0.052671,0.002103
2025-04-15 17:00:00,0.071805,0.000000,0.028350,0.000924
2025-04-15 18:00:00,0.072314,0.000000,0.034710,0.000000
...,...,...,...,...
2025-12-31 14:00:00,0.043747,0.017674,0.093780,0.022492
2025-12-31 15:00:00,0.043325,0.016510,0.097872,0.014453
2025-12-31 16:00:00,0.041725,0.017289,0.101622,0.000908


In [40]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,3.044598,1.243900,9.665484,0.776646,0.633890,0.000000,0.000000,0.055413,0.224679,0.349377
Factor 2,1.041090,0.345105,1.768875,0.198160,4.141820,1.318625,0.352748,0.083739,0.000000,0.000000
Factor 3,1.141223,5.662767,0.000000,0.151175,0.126908,0.000000,0.000000,0.003152,0.000516,0.000000
Factor 4,0.121766,0.119568,0.000000,0.069349,0.000000,1.367277,1.247841,1.197437,1.428245,1.508435


In [41]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.560682,0.124961,0.286146,0.006121,0.022089
no,0.670094,0.111437,0.177592,0.016334,0.024543
no2,0.135402,0.024485,0.839267,0.003553,-0.002707
o3,0.896494,0.106935,0.000000,0.000000,-0.003429
bin0,0.181395,0.772507,0.049446,0.000000,-0.003348
bin1,0.000000,0.765307,0.000000,0.332359,-0.097666
bin2,0.000000,0.417252,0.000000,0.618201,-0.035453
bin3,0.126145,0.124247,0.009769,0.744133,-0.004295
bin4,0.365185,0.000000,0.001142,0.633710,-0.000038
bin5,0.464131,0.000000,0.000000,0.547029,-0.011160


In [ ]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')

In [42]:
len(data)

6146